# 02 — Target Kelas Tereduksi (buang langka / gabung strain)

**Tujuan:** Mengukur seberapa banyak F1 macro rendah (0.41 di 36 kelas)
disebabkan **granularitas kelas** vs kelemahan model — dengan meredefinisi
target jadi lebih kasar dan melatih ulang LightGBM.

**PERINGATAN metodologi:** ini **mengubah definisi target**. Bukan "F1 macro
naik jadi X" — tapi "menjawab pertanyaan yang lebih kasar (25 kelas alih-alih
36) menghasilkan F1 macro X". Perbandingan ke baseline 36-kelas **apples-to-
oranges**. Kalau dipakai di skripsi harus dilaporkan jujur: *"kami memprediksi
target 25-kelas — strain minor digabung, grand slam digabung, 5NT dikecualikan"*.

**Skema yang diuji** (fitur 182 tidak berubah, split group-aware tidak
berubah — hanya kolom `target_base` yang di-remap):

| Skema | Deskripsi | ~Kelas |
|---|---|---|
| A | Baseline (36 kelas) | 36 |
| B | Buang kelas <100 sampel (`5N,1C,1D,7C,7D,7N,7S`) | 29 |
| C | Gabung grand slam `7C/7D/7H/7S/7N` → `7X` | 32 |
| D | Gabung semua strain minor (`xC`+`xD` → `xm`, level 1–6) | 30 |
| E | Penuh: C + D + buang `5N` (1C/1D ikut jadi `1m`, tidak dibuang) | 25 |

## 0. Setup

In [ ]:
import sys, json, time, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import load_splits
from src.models import LGBMModel
from src.evaluation import evaluate, print_summary

OUT = ROOT / "experiments" / "2026-08-31" / "outputs" / "reduced_class_target"
OUT.mkdir(parents=True, exist_ok=True)
CONFIG = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text())
LGBM_CFG = CONFIG["models"]["lightgbm"]

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
print("setup selesai")

## 1. Load Data + Definisi Skema Remap

In [ ]:
df_train, df_val, df_test, feature_cols, le0 = load_splits(ROOT / "data" / "processed")
feature_cols = list(feature_cols)

Xtr = df_train[feature_cols].values.astype(np.float32)
Xva = df_val[feature_cols].values.astype(np.float32)
Xte = df_test[feature_cols].values.astype(np.float32)
ttr = df_train["target_base"].astype(str).values
tva = df_val["target_base"].astype(str).values
tte = df_test["target_base"].astype(str).values
print(f"Train {Xtr.shape} | Val {Xva.shape} | Test {Xte.shape} | {df_train['target_base'].nunique()} kelas")

GRAND_SLAM = {"7C", "7D", "7H", "7S", "7N"}
MINOR = {f"{lv}{s}": f"{lv}m" for lv in "123456" for s in "CD"}   # 1C..6D -> 1m..6m
DROP_LT100 = {"5N", "1C", "1D", "7C", "7D", "7N", "7S"}

def remap(t, scheme):
    if scheme == "A_baseline":     return t
    if scheme == "B_drop_lt100":   return None if t in DROP_LT100 else t
    if scheme == "C_merge_gslam":  return "7X" if t in GRAND_SLAM else t
    if scheme == "D_merge_minor":  return MINOR.get(t, t)
    if scheme == "E_full":
        if t in GRAND_SLAM: return "7X"
        if t == "5N":       return None
        return MINOR.get(t, t)
    raise ValueError(scheme)

SCHEMES = ["A_baseline", "B_drop_lt100", "C_merge_gslam", "D_merge_minor", "E_full"]

print(f"\n{'skema':<15}{'kelas':>7}{'train':>9}{'val':>8}{'test':>8}")
for s in SCHEMES:
    yt = np.array([remap(t, s) for t in ttr], dtype=object)
    keep = yt != None  # noqa: E711
    n_cls = len(set(yt[keep]))
    yv = np.array([remap(t, s) for t in tva], dtype=object); kv = yv != None  # noqa
    ye = np.array([remap(t, s) for t in tte], dtype=object); ke = ye != None  # noqa
    print(f"{s:<15}{n_cls:>7}{keep.sum():>9}{kv.sum():>8}{ke.sum():>8}")

## 2. Retrain LightGBM per Skema — Validation Set

LightGBM `class_weight="balanced"` + hyperparameter `configs/config.yaml`
(sama seperti `notebooks/03`). Fitur 182 tidak berubah — hanya label.

In [ ]:
def run_scheme(scheme, X_eval, t_eval, split_name):
    yt = np.array([remap(t, scheme) for t in ttr], dtype=object)
    kt = yt != None  # noqa: E711
    ye = np.array([remap(t, scheme) for t in t_eval], dtype=object)
    ke = ye != None  # noqa: E711

    le = LabelEncoder().fit(yt[kt].astype(str))
    seen = set(le.classes_)
    ke = ke & np.array([str(v) in seen for v in ye])   # buang kelas eval yg tak ada di train

    ytr_enc = le.transform(yt[kt].astype(str))
    yev_enc = le.transform(ye[ke].astype(str))

    m = LGBMModel(**LGBM_CFG)
    t0 = time.time()
    m.fit(Xtr[kt], ytr_enc)
    dt = time.time() - t0
    res = evaluate(yev_enc, m.predict(X_eval[ke]), m.predict_proba(X_eval[ke]), le, model_name=scheme)
    res["n_classes"] = len(le.classes_)
    res["n_eval"] = int(ke.sum())
    res["fit_s"] = round(dt, 1)
    return m, le, res, (kt, ke)

val_results = []
for s in SCHEMES:
    _, _, r, _ = run_scheme(s, Xva, tva, "val")
    val_results.append(r)
    print(f"[{s:<14}] {r['n_classes']:>2} kelas  acc={r['accuracy']:.4f}  "
          f"f1_macro={r['f1_macro']:.4f}  f1_weighted={r['f1_weighted']:.4f}  "
          f"top3={r.get('top_3_accuracy',0):.4f}  ({r['fit_s']}s)")

val_df = pd.DataFrame([{k: r[k] for k in
    ("model", "n_classes", "n_eval", "accuracy", "f1_macro", "f1_weighted",
     "top_3_accuracy", "top_5_accuracy")} for r in val_results])
val_df.to_csv(OUT / "sweep_val.csv", index=False)
print("\n" + val_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
labels = [s.split("_", 1)[1] for s in SCHEMES]
for ax, metric, ttl in [(axes[0], "f1_macro", "F1 Macro"),
                        (axes[1], "accuracy", "Accuracy"),
                        (axes[2], "f1_weighted", "F1 Weighted")]:
    vals = [r[metric] for r in val_results]
    bars = ax.bar(labels, vals, color=["#888", "#4472C4", "#ED7D31", "#70AD47", "#C62828"])
    ax.bar_label(bars, fmt="%.3f", fontsize=9)
    ax.axhline(val_results[0][metric], color="#888", ls="--", alpha=0.6)
    ax.set_title(f"{ttl} (val)", fontweight="bold")
    ax.set_ylim(0, max(vals) * 1.2)
    ax.tick_params(axis="x", rotation=25)
plt.suptitle("Target kelas tereduksi — val set (garis putus = baseline 36 kelas)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "sweep_val.png", bbox_inches="tight")
plt.show()

## 3. F1 per-Kelas — Skema E (penuh) vs Baseline 36

In [ ]:
m_e, le_e, res_e, (kt_e, _) = run_scheme("E_full", Xva, tva, "val")
ye = np.array([remap(t, "E_full") for t in tva], dtype=object)
ke = (ye != None) & np.array([str(v) in set(le_e.classes_) for v in ye])  # noqa
f1_e = f1_score(le_e.transform(ye[ke].astype(str)), m_e.predict(Xva[ke]),
                labels=range(len(le_e.classes_)), average=None, zero_division=0)

m_a, le_a, res_a, _ = run_scheme("A_baseline", Xva, tva, "val")
f1_a = f1_score(le_a.transform(tva), m_a.predict(Xva),
                labels=range(len(le_a.classes_)), average=None, zero_division=0)

cnt_e = Counter(ye[ke].astype(str))
pc_e = pd.DataFrame({"kelas": le_e.classes_,
                     "n_val": [cnt_e[c] for c in le_e.classes_],
                     "f1": f1_e}).sort_values("f1")
pc_e.to_csv(OUT / "per_class_E.csv", index=False)
print("=== Skema E (25 kelas) — F1 per kelas (val) ===")
print(pc_e.to_string(index=False))
print(f"\nF1 macro  baseline 36 : {f1_a.mean():.4f}")
print(f"F1 macro  skema E  25 : {f1_e.mean():.4f}")
print(f"Kelas gabungan minor  : {[c for c in le_e.classes_ if c.endswith('m')]}")
print(f"F1 kelas '7X'          : {pc_e.loc[pc_e.kelas=='7X','f1'].values}")

## 4. Evaluasi Test Set — Baseline + Skema Terpilih (sekali)

In [ ]:
best_val = max(val_results[1:], key=lambda r: r["f1_macro"])  # skip A
pick = [("A_baseline", "Baseline (36 kelas)"),
        (best_val["model"], f"{best_val['model']} (F1m val terbaik)")]
if "E_full" not in [p[0] for p in pick]:
    pick.append(("E_full", "E_full (25 kelas)"))

test_rows = []
for scheme, label in pick:
    m, le, _, (kt, _) = run_scheme(scheme, Xte, tte, "test")
    ye = np.array([remap(t, scheme) for t in tte], dtype=object)
    ke = (ye != None) & np.array([str(v) in set(le.classes_) for v in ye])  # noqa
    yenc = le.transform(ye[ke].astype(str))
    r = evaluate(yenc, m.predict(Xte[ke]), m.predict_proba(Xte[ke]), le, model_name=label)
    test_rows.append({"skema": label, "n_classes": len(le.classes_), "n_test": int(ke.sum()),
                      "accuracy": r["accuracy"], "f1_macro": r["f1_macro"],
                      "f1_weighted": r["f1_weighted"],
                      "top_3": r["top_3_accuracy"], "top_5": r["top_5_accuracy"]})
    print_summary(r)

test_df = pd.DataFrame(test_rows)
test_df.to_csv(OUT / "test_comparison.csv", index=False)
print("\n" + test_df.to_string(index=False))

summary = {
    "purpose": "reduced-class target (drop rare / merge strains) vs 36-class baseline",
    "note": "CHANGES THE PREDICTION TARGET - not a like-for-like F1 macro improvement",
    "schemes": {s: "see notebook" for s in SCHEMES},
    "val": val_df.to_dict(orient="records"),
    "test": test_rows,
}
with open(OUT / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print("\nsummary.json disimpan.")

## 5. Kesimpulan

*(diisi setelah eksekusi — lihat `summary.json` + `test_comparison.csv`)*